# Session 9: AI Agents — Concepts and Patterns

This notebook covers:
1. **Agentic RAG** — the bridge from Session 8
2. **LLM vs. Agent** — what changes when you add a control loop
3. **Workflow patterns** — prompt chaining, routing, parallelization, evaluator-optimizer
4. **MCP** — the universal wiring layer for agent tools

We build a small FAISS index inline so everything is self-contained.

In [5]:
import os
import sys
import logging
import numpy as np
import faiss
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from dotenv import load_dotenv

# Suppress verbose HTTP logs from the OpenAI client
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

load_dotenv()
client = OpenAI()
print("Setup complete")

Setup complete


## 1. Agentic RAG — The Bridge from Session 8

In Session 8 we built **single-shot RAG**:
```
query → embed → retrieve → generate → done
```

**Agentic RAG** adds a loop around that pipeline. But there are two very different ways
to implement that loop — and the difference is the central lesson of this session.

**Version 1 (this section):** The programmer writes the loop. We decide when to check
completeness, when to rewrite the query, when to stop. The LLM answers questions,
but the control flow is entirely ours.

**Version 2 (notebook 02):** We give the LLM a `retrieve` tool and let it run its own loop.
The LLM decides when to search, what to search for, and when it has enough to answer.

Version 1 is a **workflow**. Version 2 is an **agent**.
We build Version 1 first so the difference is concrete when we get to Version 2.

We start by building a small in-memory FAISS index over 10 sample texts.

In [2]:
SAMPLE_TEXTS = [
    "Retrieval-Augmented Generation (RAG) combines a retrieval system with an LLM to ground answers in documents.",
    "FAISS is a library for efficient similarity search over dense vectors, built by Meta.",
    "Cosine similarity measures the angle between two vectors, returning a value between -1 and 1.",
    "An embedding is a dense numerical representation of text that captures semantic meaning.",
    "BM25 is a lexical search algorithm scoring by term frequency and inverse document frequency.",
    "The OpenAI Agents SDK uses Agent, Runner, function_tool, and handoffs as its core primitives.",
    "Agentic RAG lets an LLM iteratively retrieve and refine its answer rather than retrieving once.",
    "MCP (Model Context Protocol) is an open standard for connecting AI models to tools and data sources.",
    "Prompt chaining passes the output of one LLM call as the input to the next in a pipeline.",
    "The compound accuracy problem: a 10-step workflow at 85% accuracy per step succeeds only 20% of the time.",
]

def embed_texts(texts: list[str]) -> np.ndarray:
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([item.embedding for item in response.data], dtype=np.float32)

def embed_query(query: str) -> np.ndarray:
    response = client.embeddings.create(model="text-embedding-3-small", input=[query])
    return np.array(response.data[0].embedding, dtype=np.float32)

embeddings = embed_texts(SAMPLE_TEXTS)
faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
faiss_index.add(embeddings)
print(f"FAISS index ready: {faiss_index.ntotal} vectors, dimension {embeddings.shape[1]}")

FAISS index ready: 10 vectors, dimension 1536


### The Agentic RAG Loop

Four helper functions, each doing one job:
- `retrieve()` — embed the query and find nearest chunks in FAISS
- `generate()` — call the LLM with retrieved context
- `is_complete()` — ask the LLM if the answer is sufficient
- `rewrite_query()` — ask the LLM what to search for next

In [3]:
def retrieve(query: str, top_k: int = 2) -> list[str]:
    """Find the top-k most relevant chunks for a query."""
    q_vec = embed_query(query).reshape(1, -1)
    _, indices = faiss_index.search(q_vec, top_k)
    return [SAMPLE_TEXTS[i] for i in indices[0]]

def generate(question: str, context_chunks: list[str]) -> str:
    """Generate an answer grounded in the retrieved context."""
    context = "\n\n".join(context_chunks)
    return client.responses.create(
        model="gpt-4o-mini",
        instructions=(
            "Answer using only the provided context. "
            "If context is insufficient, say exactly what information is missing."
        ),
        input=f"Context:\n{context}\n\nQuestion: {question}"
    ).output_text

def is_complete(question: str, answer: str) -> bool:
    """Ask the LLM whether the answer fully addresses the original question."""
    reply = client.responses.create(
        model="gpt-4o-mini",
        instructions="Reply only 'yes' or 'no'. Does this answer fully address the question?",
        input=f"Question: {question}\n\nAnswer: {answer}"
    ).output_text.strip().lower()
    return reply.startswith("yes")

def rewrite_query(original_question: str, partial_answer: str) -> str:
    """Generate a better search query to find what's still missing."""
    # Simplification: always rewrites from the original question + current partial answer.
    # A production loop would also pass the current query to avoid regenerating the same rewrite.
    return client.responses.create(
        model="gpt-4o-mini",
        instructions="Write a short search query (under 10 words) to find the missing information.",
        input=f"Original question: {original_question}\nPartial answer: {partial_answer}"
    ).output_text.strip()

In [4]:
context_chunks = retrieve(query="What is Retrieval-Augmented Generation?", top_k=2)
print("Initial retrieved context:")
for i, chunk in enumerate(context_chunks, 1):
    print(f"{i}. {chunk}")

Initial retrieved context:
1. Retrieval-Augmented Generation (RAG) combines a retrieval system with an LLM to ground answers in documents.
2. Agentic RAG lets an LLM iteratively retrieve and refine its answer rather than retrieving once.


In [5]:
response = generate(
    question="What is Retrieval-Augmented Generation?",
    context_chunks=context_chunks
    )

print("\nGenerated answer:")
print(response)


Generated answer:
Retrieval-Augmented Generation (RAG) combines a retrieval system with a large language model (LLM) to produce answers that are grounded in relevant documents.


In [6]:
is_complete_answer = is_complete(
    question="What is Retrieval-Augmented Generation?",
    answer=response
)
print(f"\nIs the answer complete? {'Yes' if is_complete_answer else 'No'}")


Is the answer complete? Yes


In [7]:
rewritten_query = rewrite_query(
    original_question="What is Retrieval-Augmented Generation?",
    partial_answer=response
)
print(f"\nRewritten query to find missing info: {rewritten_query}")



Rewritten query to find missing info: "How does Retrieval-Augmented Generation work?"


In [8]:
def agentic_rag(question: str, max_iterations: int = 3) -> str:
    """
    Agentic RAG control loop:
      1. Retrieve relevant chunks for the current query
      2. Generate an answer from accumulated context
      3. If complete → return. If not → rewrite query and loop.
    """
    query = question
    accumulated_context: list[str] = []

    print(f"Question: {question}\n{'='*60}")

    for i in range(max_iterations):
        print(f"\n[Iteration {i+1}] Searching for: {query}")
        new_chunks = [c for c in retrieve(query) if c not in accumulated_context]
        accumulated_context.extend(new_chunks)
        print(f"  {len(new_chunks)} new chunk(s) retrieved (total: {len(accumulated_context)})")

        answer = generate(question, accumulated_context)
        preview = answer[:120] + ("..." if len(answer) > 120 else "")
        print(f"  Answer: {preview}")

        if is_complete(question, answer):
            print(f"\n✓ Complete after {i+1} iteration(s)")
            return answer

        query = rewrite_query(question, answer)
        print(f"  → Rewriting query: {query}")

    print(f"\n⚠ Reached max iterations ({max_iterations})")
    return answer

answer = agentic_rag(
    "do penguins have knees?"
    #"How does FAISS support semantic search, and what similarity metric does it use?"
)
print(f"\nFinal answer:\n{answer}")

Question: do penguins have knees?

[Iteration 1] Searching for: do penguins have knees?
  2 new chunk(s) retrieved (total: 2)
  Answer: The provided context does not include any information regarding whether penguins have knees.
  → Rewriting query: "Do penguins have knees anatomy details?"

[Iteration 2] Searching for: "Do penguins have knees anatomy details?"
  0 new chunk(s) retrieved (total: 2)
  Answer: The provided context does not contain any information about penguins or their anatomy.
  → Rewriting query: "Do penguins have knees anatomy facts?"

[Iteration 3] Searching for: "Do penguins have knees anatomy facts?"
  1 new chunk(s) retrieved (total: 3)
  Answer: The information provided does not include details about penguins or their anatomy. Please provide more context or ask an...
  → Rewriting query: Do penguins have knees in their anatomy?

⚠ Reached max iterations (3)

Final answer:
The information provided does not include details about penguins or their anatomy. Please p

In [9]:
answer = agentic_rag(
    "What is the compound accuracy problem in multi-step workflows, and how does agentic RAG help mitigate it?"
)
print(f"\nFinal answer:\n{answer}")

Question: What is the compound accuracy problem in multi-step workflows, and how does agentic RAG help mitigate it?

[Iteration 1] Searching for: What is the compound accuracy problem in multi-step workflows, and how does agentic RAG help mitigate it?
  2 new chunk(s) retrieved (total: 2)
  Answer: The compound accuracy problem in multi-step workflows refers to the cumulative effect of individual step accuracies. In ...

✓ Complete after 1 iteration(s)

Final answer:
The compound accuracy problem in multi-step workflows refers to the cumulative effect of individual step accuracies. In the given example, if each of the 10 steps has an accuracy of 85%, the overall success rate plummets to only 20% due to the compounding inaccuracies across steps.

Agentic RAG helps mitigate this problem by allowing a large language model (LLM) to iteratively retrieve and refine its answers. Instead of relying on a single retrieval, it enhances accuracy by continuously refining responses through multiple 

### What the programmer still controls

Look at `agentic_rag()` and ask: *who is making each decision?*

| Decision | Who makes it? |
|---|---|
| How many times to try | Programmer (`max_iterations = 3`) |
| Whether to check completeness | Programmer (calls `is_complete()` every iteration) |
| Whether to rewrite the query | Programmer (always calls `rewrite_query()` if not complete) |
| When to stop | Programmer (first `is_complete()` → True, or max reached) |

The LLM is answering questions inside each step — but the **control flow is entirely ours**.
This is a workflow that *simulates* agentic behaviour, not a true agent.

**A true agent:** give the LLM a `retrieve` tool and let `Runner` manage the loop.
The LLM then decides *if* to retrieve, *what* to search for, and *when* it has enough.
We build that in notebook 02.

## 2. LLM vs. Agent — What Changes When You Add a Loop

| | LLM | Agent |
|---|---|---|
| Autonomy | Reacts to prompts | Acts toward a goal |
| Tool use | Needs explicit calling | Decides when to call tools |
| Memory | Stateless | Accumulates context |
| Workflow | Single-step | Multi-step, iterative |

The core difference: **an agent decides its own next action**.
A static LLM call is determined entirely by the caller.

In [10]:
QUESTION = (
    "What is cosine similarity and what is 0.85 raised to the power of 10? "
    "Also, what does FAISS stand for?"
    "Is iPhone 18 released yet?"
)

# Static LLM: one call, no tools, relies entirely on training data
print("=== Static LLM (one call, no tools) ===")
static_answer = client.responses.create(
    model="gpt-4o-mini",
    instructions="Answer the question concisely.",
    input=QUESTION
).output_text
print(static_answer)

print("\n" + "="*60)
print("=== What an agent would do differently ===")
print("""
1. Recognise this question has three independent sub-tasks
2. Call a search tool for 'cosine similarity' → retrieve definition
3. Call a calculator tool for 0.85 ** 10 → exact result
4. Call a search tool for 'FAISS' → retrieve acronym
5. Combine results into one coherent answer

The agent decides WHEN and WHICH tools to call.
We build this in notebook 02 using the OpenAI Agents SDK.
""")

=== Static LLM (one call, no tools) ===
Cosine similarity is a measure used to determine how similar two vectors are, calculated as the cosine of the angle between them. It ranges from -1 to 1, where 1 indicates maximum similarity.

0.85 raised to the power of 10 is approximately 0.19687.

FAISS stands for Facebook AI Similarity Search.

As of now, the iPhone 18 has not been released. The latest models available are from the iPhone 15 series.

=== What an agent would do differently ===

1. Recognise this question has three independent sub-tasks
2. Call a search tool for 'cosine similarity' → retrieve definition
3. Call a calculator tool for 0.85 ** 10 → exact result
4. Call a search tool for 'FAISS' → retrieve acronym
5. Combine results into one coherent answer

The agent decides WHEN and WHICH tools to call.
We build this in notebook 02 using the OpenAI Agents SDK.



In [11]:
print("=== Static LLM (one call, with web search tool) ===")
static_answer = client.responses.create(
    model="gpt-4o-mini",
    instructions="Answer the question concisely.",
    input=QUESTION,
    tools=[{ "type": "web_search" }],
    tool_choice="auto"
).output_text
print(static_answer)

=== Static LLM (one call, with web search tool) ===
**Cosine Similarity** is a metric used to measure how similar two vectors are, by calculating the cosine of the angle between them. It ranges from -1 to 1, where 1 means the vectors are identical, and 0 indicates orthogonality (no similarity).

**0.85 raised to the power of 10** is approximately **0.1969**.

**FAISS** stands for **Facebook AI Similarity Search**, a library developed by Facebook for efficient similarity search and clustering of dense vectors.

As for the **iPhone 18**, it has not been released yet. The latest iPhone models announced were the iPhone 15 series.


## 3. Workflow Patterns

Six patterns from Anthropic's [Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents) article.  
We implement four as working code examples.

---

### Pattern 1: Prompt Chaining

Output of one LLM call feeds the next. Good for tasks with clear sequential stages.
Optional gate checks between steps let you exit early if a condition is not met.

![Prompt Chaining](https://www-cdn.anthropic.com/images/4zrzovbb/website/7418719e3dab222dccb379b8879e1dc08ad34c78-2401x1000.png)

**Our demo:** raw notes → extract key points → formalise → add headline


In [12]:
def prompt_chain(raw_notes: str) -> dict:
    """Three-step chain: extract → formalise → headline."""
    print("Step 1: Extract key points")
    key_points = client.responses.create(
        model="gpt-4o-mini",
        instructions="Extract exactly 3 key points as a bullet list.",
        input=raw_notes
    ).output_text
    print(key_points)

    print("Step 2: Rewrite in formal language")
    formal = client.responses.create(
        model="gpt-4o-mini",
        instructions="Rewrite these bullet points in formal business language.",
        input=key_points
    ).output_text
    print(formal)

    print("Step 3: Generate a one-sentence headline")
    headline = client.responses.create(
        model="gpt-4o-mini",
        instructions="Write a single-sentence executive headline summarising these points.",
        input=formal
    ).output_text
    print(f"\n📰 Headline: {headline}")

    return {"key_points": key_points, "formal": formal, "headline": headline}

RAW_NOTES = """
Session 9 covered AI agents. We learned that agents use a control loop to decide
when and how to use tools. We also covered MCP, which standardises how agents
connect to external tools. The hands-on session used the OpenAI Agents SDK to
build a tool-using agent with FAISS-based search.
"""

chain_result = prompt_chain(RAW_NOTES)

Step 1: Extract key points
- AI agents utilize a control loop to determine the timing and method of tool usage.
- MCP (Multi-Channel Protocol) standardizes connections between agents and external tools.
- The practical session involved creating a tool-using agent using the OpenAI Agents SDK and FAISS-based search.
Step 2: Rewrite in formal language
- Artificial Intelligence agents employ a control loop to ascertain the timing and methodology for utilizing tools.
- The Multi-Channel Protocol (MCP) standardizes the connections between agents and external tools.
- The practical session focused on developing a tool-utilizing agent through the OpenAI Agents Software Development Kit (SDK) and integrating FAISS-based search capabilities.
Step 3: Generate a one-sentence headline

📰 Headline: Artificial Intelligence agents leverage a control loop and the Multi-Channel Protocol (MCP) to efficiently utilize tools, with practical sessions focusing on developing tool-utilizing agents through the Op

### Pattern 2: Routing

Classify the input first, then send it to the right specialised handler.
Used in customer service, triage systems, and intent-based pipelines.

![Routing](https://www-cdn.anthropic.com/images/4zrzovbb/website/5c0c0e9fe4def0b584c04d37849941da55e5e71c-2401x1000.png)

**Our demo:** billing / technical / returns / other — one LLM classifies, then a handler responds.


In [13]:
HANDLERS = {
    "billing":   lambda q: f"[Billing Agent] Processing payment query: {q}",
    "technical": lambda q: f"[Tech Support] Troubleshooting: {q}",
    "returns":   lambda q: f"[Returns Agent] Processing return request: {q}",
    "other":     lambda q: f"[General Agent] Answering: {q}",
}

def route_and_handle(query: str) -> str:
    intent = client.responses.create(
        model="gpt-4o-mini",
        instructions=(
            "Classify this customer query as exactly one of: "
            "billing, technical, returns, other. Reply with one word only."
        ),
        input=query
    ).output_text.strip().lower()

    category = intent if intent in HANDLERS else "other"
    response = HANDLERS[category](query)
    print(f"Query:    {query}")
    print(f"Routed to: {category}")
    print(f"Response: {response}\n")
    return response

TEST_QUERIES = [
    "My invoice shows double the amount I should be paying",
    "The app keeps crashing when I open the settings page",
    "I want to return the headphones I bought last week",
    "What time does your support team work?",
]

for q in TEST_QUERIES:
    route_and_handle(q)

Query:    My invoice shows double the amount I should be paying
Routed to: billing
Response: [Billing Agent] Processing payment query: My invoice shows double the amount I should be paying

Query:    The app keeps crashing when I open the settings page
Routed to: technical
Response: [Tech Support] Troubleshooting: The app keeps crashing when I open the settings page

Query:    I want to return the headphones I bought last week
Routed to: returns
Response: [Returns Agent] Processing return request: I want to return the headphones I bought last week

Query:    What time does your support team work?
Routed to: other
Response: [General Agent] Answering: What time does your support team work?



### Pattern 3: Parallelization

Run independent LLM calls concurrently — either fan-out to specialised workers (sectioning)
or send the same input to multiple models and aggregate (voting).

![Parallelization](https://www-cdn.anthropic.com/images/4zrzovbb/website/406bb032ca007fd1624f261af717d70e6ca86286-2401x1000.png)

**Our demo:** three paper abstracts summarised in parallel with `ThreadPoolExecutor`.


In [14]:
PAPERS = [
    {
        "title": "RAG (Lewis et al., 2020)",
        "text": (
            "Retrieval-Augmented Generation combines a dense retriever with a seq2seq generator "
            "to ground answers in retrieved documents, reducing hallucination on open-domain QA."
        ),
    },
    {
        "title": "Sentence-BERT (Reimers & Gurevych, 2019)",
        "text": (
            "Sentence-BERT uses siamese BERT networks with mean pooling to produce fixed-length "
            "sentence embeddings suited for semantic similarity via cosine distance."
        ),
    },
    {
        "title": "FAISS (Johnson et al., 2019)",
        "text": (
            "FAISS provides efficient exact and approximate nearest-neighbor search over "
            "billion-scale dense vectors using GPU-accelerated IVF and HNSW index types."
        ),
    },
]

def summarise_one(paper: dict) -> tuple[str, str]:
    summary = client.responses.create(
        model="gpt-4o-mini",
        instructions="Summarise in exactly one sentence for a Master's student.",
        input=paper["text"]
    ).output_text.strip()
    return paper["title"], summary

print("Summarising 3 papers in parallel...\n")
with ThreadPoolExecutor(max_workers=3) as pool:
    results = list(pool.map(summarise_one, PAPERS))

for title, summary in results:
    print(f"📄 {title}")
    print(f"   {summary}\n")

Summarising 3 papers in parallel...

📄 RAG (Lewis et al., 2020)
   Retrieval-Augmented Generation integrates a dense retriever with a sequence-to-sequence generator to enhance the accuracy of responses in open-domain question answering by grounding them in relevant retrieved documents, thereby minimizing hallucination.

📄 Sentence-BERT (Reimers & Gurevych, 2019)
   Sentence-BERT leverages siamese BERT architectures with mean pooling to generate fixed-length sentence embeddings optimized for semantic similarity measurements using cosine distance.

📄 FAISS (Johnson et al., 2019)
   FAISS facilitates high-performance exact and approximate nearest-neighbor search for billion-scale dense vectors by leveraging GPU-accelerated Index Flat (IVF) and Hierarchical Navigable Small World (HNSW) index structures.



### Pattern 4: Evaluator-Optimizer

One LLM generates output; another scores it and provides feedback.
The generator improves until the score is high enough or a max-rounds limit is hit.

![Evaluator-Optimizer](https://www-cdn.anthropic.com/images/4zrzovbb/website/14f51e6406ccb29e695da48b17017e899a6119c7-2401x1000.png)

Used in: content pipelines, coding agent loops, automated essay refinement.


In [15]:
def evaluator_optimizer(task: str, target_score: int = 8, max_rounds: int = 3) -> str:
    """Generate → evaluate → improve loop. Stop when score ≥ target_score."""
    print(f"Task: {task}\n{'='*60}")

    draft = client.responses.create(
        model="gpt-4o-mini",
        instructions=
        "Complete the task by suggesting ideas and structuring your thoughts. "
        "Your first attempt should always be raw and unpolished to structure your thoughts. "
        "Evaluator will provide you with feedback to validate your thoughts.",
        input=task
    ).output_text

    for round_num in range(1, max_rounds + 1):
        print(f"\n[Round {round_num}] Draft preview:")
        print(draft[:180] + ("..." if len(draft) > 180 else ""))

        evaluation = client.responses.create(
            model="gpt-4o-mini",
            instructions=(
                "Evaluate this output on a scale of 1–10. "
                "Provide specific feedback on how to improve it. "
                "Reply in exactly this format, nothing else:\n"
                "SCORE: <number>\n"
                "FEEDBACK: <one sentence of specific improvement needed>"
            ),
            input=f"Task: {task}\n\nOutput:\n{draft}"
        ).output_text

        lines = {
            line.split(":", 1)[0].strip(): line.split(":", 1)[1].strip()
            for line in evaluation.splitlines()
            if ":" in line
        }
        score = int(lines.get("SCORE", "0").split()[0])
        feedback = lines.get("FEEDBACK", "No specific feedback")

        print(f"Score: {score}/10 | Feedback: {feedback}")

        if score >= target_score:
            print(f"\n✓ Accepted at round {round_num} with score {score}/10")
            return draft

        draft = client.responses.create(
            model="gpt-4o-mini",
            instructions="Consider the feedback and improve the output based on the feedback. Keep the same format."
            "explain how you improved based on the feedback in your response.",
            input=f"Task: {task}\n\nCurrent output:\n{draft}\n\nFeedback: {feedback}"
        ).output_text

    print(f"\n⚠ Reached max rounds. Returning best draft (score {score}/10)")
    return draft

final = evaluator_optimizer(
    "Explain what a vector embedding is to a first-year data science student. Use one concrete analogy.",
    target_score=9,
    max_rounds=5
)
print(f"\nFinal output:\n{final}")

Task: Explain what a vector embedding is to a first-year data science student. Use one concrete analogy.

[Round 1] Draft preview:
Sure! Let's break it down simply. 

A vector embedding is like translating something complex into a simpler form that preserves its important features. 

### Analogy: A Recipe Book...
Score: 7/10 | Feedback: The analogy could be clearer; consider explicitly linking how the summary note (the embedding) retains features of the original recipes (the data) rather than just summarizing their essence.

[Round 2] Draft preview:
Sure! I'll clarify the analogy to emphasize how the summary note retains features of the original recipes and directly connects to the concept of vector embeddings.

---

### Expla...
Score: 8/10 | Feedback: To improve, simplify the language further and ensure the analogy is succinct, focusing solely on the essence of vector embeddings without overloading with extra details.

[Round 3] Draft preview:
Sure! Based on your feedback, I’ve simpl

### Aside: where structured output would improve this

The evaluator currently returns free text in a format we parse manually:
```
SCORE: 7
FEEDBACK: Add a concrete analogy
```

This is fragile — the model might vary the format and our `split(":")` breaks.
The robust version uses a Pydantic model as the response schema:

```python
class Evaluation(BaseModel):
    score: int        # 1–10
    feedback: str     # one sentence of specific improvement

# Then parse with:
evaluation = client.responses.parse(
    model="gpt-4o-mini",
    text={"format": {"type": "json_schema", "schema": Evaluation.model_json_schema(), "strict": True}},
    input=f"Evaluate: {draft}"
)
score = evaluation.score     # int, guaranteed
feedback = evaluation.feedback  # str, guaranteed
```

We keep the string-parsing version here to keep the pattern readable.
In production, always use structured output for anything you need to act on programmatically.

## 4. Model Context Protocol (MCP)

### The Problem: M × N Integrations

Before MCP, every AI application needed custom code to talk to every tool:
- Claude talking to GitHub = custom integration
- Claude talking to Notion = another custom integration
- ChatGPT talking to GitHub = another custom integration
- ...M applications × N tools = M×N bespoke integrations

### The Solution: M + N

MCP defines a **single standard interface** (JSON-RPC based):
- Each AI **client** (Claude, ChatGPT, Cursor, VS Code) implements MCP once
- Each **tool server** (GitHub, Notion, Postgres, your own app) implements MCP once
- They connect without any custom glue code

Analogy: **"USB-C for language models"** — one standard plug for any context source.

### Adoption (April 2026)
- Introduced by Anthropic, November 2024
- Donated to Linux Foundation (Agentic AI Foundation), April 2026
- 97 million monthly SDK downloads
- 10,000+ public MCP servers
- Supported by: Claude, ChatGPT, Cursor, VS Code, GitHub Copilot, AWS, Azure, GCP

```
BEFORE MCP: M apps × N tools = M×N integrations
─────────────────────────────────────────────────────
  Claude ──── custom ──── GitHub API
  Claude ──── custom ──── Notion API
  Claude ──── custom ──── Postgres
  ChatGPT ─── custom ──── GitHub API
  ChatGPT ─── custom ──── Notion API
  Cursor ──── custom ──── GitHub API
  (every pair needs its own code)

AFTER MCP: M+N — each side implements once
─────────────────────────────────────────────────────
  Claude  ──────┐
  ChatGPT ──────┼── MCP Client ══ MCP Protocol ══ GitHub MCP Server
  Cursor  ──────┘                               ╠══ Notion MCP Server
                                                ╚══ Your App MCP Server
```

### Building an MCP Server

The `mcp` Python package lets you define a server in a few lines.
Once running, any MCP-compatible client can call your tools automatically —
no additional integration code needed.

In [16]:
from mcp.server.fastmcp import FastMCP

# Define an MCP server
mcp_server = FastMCP("ANLP Course Server")

@mcp_server.tool()
def get_session_topics(session_number: int) -> str:
    """Get the topics covered in a given ANLP course session."""
    topics = {
        7: "LLM Deep Dive, Prompting Techniques (CoT, few-shot), LLM Evaluation (BLEU, ROUGE, LLM-as-judge)",
        8: "LLM API Access, RAG, FAISS, Streaming, Gradio/Streamlit, Tool Use",
        9: "AI Agents, Workflow Patterns, MCP, OpenAI Agents SDK",
    }
    return topics.get(session_number, f"Session {session_number} not in course")

@mcp_server.tool()
def search_glossary(term: str) -> str:
    """Look up a term in the ANLP course glossary."""
    glossary = {
        "rag": "Retrieval-Augmented Generation — grounding LLM answers in retrieved documents",
        "embedding": "A dense numerical vector representing text meaning",
        "faiss": "Facebook AI Similarity Search — fast vector similarity search library",
        "mcp": "Model Context Protocol — open standard for connecting AI models to tools",
        "agent": "An LLM extended with tools that can take multi-step actions toward a goal",
        "bm25": "A lexical search algorithm scoring by term frequency and document rarity",
        "cosine similarity": "A measure of angle between two vectors, bounded between -1 and 1",
    }
    return glossary.get(term.lower(), f"'{term}' not in glossary — try a related term")

# Test the tools directly (they are just Python functions)
print(get_session_topics(8))
print()
print(search_glossary("mcp"))

LLM API Access, RAG, FAISS, Streaming, Gradio/Streamlit, Tool Use

Model Context Protocol — open standard for connecting AI models to tools


### Connecting an Agent to the MCP Server

The real power of MCP is not just defining tools — it is that **any MCP-compatible client
can discover and call those tools automatically**, with no integration code.

The server lives in `mcp_course_server.py` (same folder as this notebook).
Below we use the OpenAI Agents SDK's `MCPServerStdio` to:
1. Launch `mcp_course_server.py` as a subprocess — this is the actual MCP protocol over stdio
2. Create an agent that **auto-discovers** `get_session_topics` and `search_glossary`
3. Ask a multi-part question — the agent decides which tools to call

No `@function_tool` decorators. No explicit tool list. The tools come entirely from the server.

In [1]:
import sys
from pathlib import Path

# Path to the standalone MCP server in this folder
_server_path = Path("mcp_course_server.py").resolve()
print(f"Server: {_server_path}")

Server: /home/dave/Development/UTS/nlp/anlpaut2026/material/Session 9/notebooks/mcp_course_server.py


### How to run this as a real MCP server

Add `mcp_server.run()` at the end of a Python file and run it.
Then add it to any MCP client's config. Example for **Claude Desktop** or **Gitbhub Copilot**:

```json
{
  "mcpServers": {
    "anlp": {
      "command": "python",
      "args": ["path/to/your_server.py"]
    }
  }
}
```

Once connected, Claude Desktop can call `get_session_topics()` and
`search_glossary()` from any conversation — with no extra code on Claude's side.

The same server also works with **Cursor**, **VS Code**, and **GitHub Copilot**
without changing a single line of the server implementation.

In [2]:
import json
print(json.dumps({"command": sys.executable, "args": [str(_server_path)]}, indent=2))

{
  "command": "/home/dave/Development/UTS/nlp/anlpaut2026/.venv/bin/python",
  "args": [
    "/home/dave/Development/UTS/nlp/anlpaut2026/material/Session 9/notebooks/mcp_course_server.py"
  ]
}


In [3]:
from agents import Agent, Runner
from mcp_notebook_helpers import NotebookSafeMCPServerStdio

In [7]:


async def run_course_bot():
    # NotebookSafeMCPServerStdio launches mcp_course_server.py as a subprocess.
    # On Windows notebooks, it avoids piping stderr into ipykernel's stream
    # object, which does not expose a real fileno() for subprocess.Popen().
    # and speaks the MCP protocol — the agent's tool list is populated
    # entirely from the server, no @function_tool decorators needed.
    async with NotebookSafeMCPServerStdio(
        params={"command": sys.executable, "args": [str(_server_path)]},
        cache_tools_list=True,
    ) as mcp:
        # Show which tools the agent discovered automatically
        tools = await mcp.list_tools()
        print("Tools discovered from MCP server:")
        for t in tools:
            print(f"  - {t.name}: {t.description}")
        print()

        course_bot = Agent(
            name="Course Bot",
            instructions=(
                "You are a helpful ANLP course assistant. "
                "Use your tools to answer questions about course sessions and terminology."
            ),
            mcp_servers=[mcp],
            model="gpt-4o-mini",
        )

        result = await Runner.run(
            course_bot,
            "What topics are covered in Sessions 8 and 9? Also, what does 'embedding' mean?",
        )
        print(result.final_output)

await run_course_bot()

Tools discovered from MCP server:
  - get_session_topics: Get the topics covered in a given ANLP course session.
  - search_glossary: Look up a term in the ANLP course glossary.



OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


### Topics Covered

**Session 8:**
- LLM API Access
- RAG (Retrieval-Augmented Generation)
- FAISS (Facebook AI Similarity Search)
- Streaming
- Gradio/Streamlit
- Tool Use

**Session 9:**
- AI Agents
- Workflow Patterns
- MCP (Multi-Component Processing)
- OpenAI Agents SDK

### Definition of 'Embedding'
**Embedding** refers to a dense numerical vector that represents the meaning of text. These vectors are used in various natural language processing tasks to capture semantic relationships between words and phrases.


## Summary

| Concept | What it does |
|---|---|
| **Agentic RAG** | LLM controls its own retrieval loop — retrieves, checks, rewrites, retrieves again |
| **Prompt chaining** | Sequential LLM calls; output of each step is input to the next |
| **Routing** | Classify input → send to the right specialist |
| **Parallelization** | Run independent LLM calls concurrently |
| **Evaluator-optimizer** | Generate → score → improve until quality threshold is met |
| **MCP** | Standard protocol so any agent can call any tool with one implementation |

**Next: notebook 02** — build real agents using the OpenAI Agents SDK.